# UAV Crop-Weed Segmentation

See [project_plan.md](project_plan.md) for the project definition and task list. This notebook contains code, outputs, and concise technical explanations in English. Read and verify all code before submission.

## 1. Data Acquisition and Source Inspection

### 1.2 Data Acquisition

Reuse the pinned local archive when available; otherwise download it. Verify its checksum and extract original RGB images, masks, and dataset documentation only.

In [1]:
from pathlib import Path, PurePosixPath
import hashlib
import shutil
import urllib.request
import zipfile

DATASET_URL = "https://www.kaggle.com/datasets/bouhadjer/crop-weed-segmentation-uav-rgb-indices"
DOWNLOAD_URL = "https://www.kaggle.com/api/v1/datasets/download/bouhadjer/crop-weed-segmentation-uav-rgb-indices"
EXPECTED_SHA256 = "1cd9eddab9c5e132f679f0b223591c2ef549282920b447a693525f973e4775e3"

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "project_plan.md").exists():
    PROJECT_ROOT = PROJECT_ROOT / "final-project"
if not (PROJECT_ROOT / "project_plan.md").exists():
    raise FileNotFoundError("Run from the project folder or the course repository root.")

DOWNLOAD_DIR = PROJECT_ROOT / "data" / "downloads"
RAW_DIR = PROJECT_ROOT / "data" / "raw"
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
ARCHIVE_PATH = DOWNLOAD_DIR / "dataset.zip"

if ARCHIVE_PATH.exists():
    print("Using the existing local archive.")
else:
    temporary_path = ARCHIVE_PATH.with_suffix(".zip.part")
    request = urllib.request.Request(DOWNLOAD_URL, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(request, timeout=120) as response:
        with temporary_path.open("wb") as destination:
            shutil.copyfileobj(response, destination)
    with temporary_path.open("rb") as handle:
        downloaded_hash = hashlib.file_digest(handle, "sha256").hexdigest()
    if downloaded_hash != EXPECTED_SHA256:
        raise ValueError("Downloaded archive differs from the pinned snapshot. Review the dataset version before proceeding.")
    temporary_path.replace(ARCHIVE_PATH)
    print("Downloaded the dataset archive.")

with ARCHIVE_PATH.open("rb") as handle:
    archive_hash = hashlib.file_digest(handle, "sha256").hexdigest()
if archive_hash != EXPECTED_SHA256:
    raise ValueError("Archive checksum mismatch. Do not use this archive without review.")

original_count = 0
document_count = 0
with zipfile.ZipFile(ARCHIVE_PATH) as archive:
    for member in archive.infolist():
        if member.is_dir():
            continue
        path = PurePosixPath(member.filename)
        is_original = path.parent.name in {"img", "msk"}
        is_document = path.parent == PurePosixPath("Dataset") and path.suffix == ".txt"
        if not (is_original or is_document):
            continue
        relative = path.relative_to("Dataset")
        if ".." in relative.parts or relative.is_absolute():
            raise ValueError("Unsafe archive path.")
        target = RAW_DIR.joinpath(*relative.parts)
        content = archive.read(member)
        if target.exists():
            if target.read_bytes() != content:
                raise ValueError(f"Existing extracted file differs from the archive: {target}")
        else:
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(content)
        original_count += int(is_original)
        document_count += int(is_document)

print("Dataset:", DATASET_URL)
print("Archive:", ARCHIVE_PATH.relative_to(PROJECT_ROOT))
print("Archive bytes:", ARCHIVE_PATH.stat().st_size)
print("SHA-256:", archive_hash)
print("Original RGB and mask files:", original_count)
print("Dataset documentation files:", document_count)
print("Raw data:", RAW_DIR.relative_to(PROJECT_ROOT))


Using the existing local archive.
Dataset: https://www.kaggle.com/datasets/bouhadjer/crop-weed-segmentation-uav-rgb-indices
Archive: data/downloads/dataset.zip
Archive bytes: 2590700651
SHA-256: 1cd9eddab9c5e132f679f0b223591c2ef549282920b447a693525f973e4775e3
Original RGB and mask files: 42
Dataset documentation files: 3
Raw data: data/raw


## 2. Source-Image Splitting

## 3. Data Exploration and Preprocessing

## 4. Baseline Models

## 5. U-Net Training

## 6. Evaluation, Ablation, and Error Analysis

## 7. Reproducibility and Report Outputs

## 8. Demonstration